## 0. 加载 Qwen 模型

本 Notebook 使用 **Qwen2.5-7B-Instruct**（通过 ModelScope 加载）作为真实 LLM 后端，替代原有的模拟 LLM。

> **低显存备选**：如果 GPU 显存不足，可将模型 ID 替换为 `Qwen/Qwen2.5-3B-Instruct`。

### 安装依赖（首次运行需要）

```bash
pip install modelscope torch transformers
```


In [ ]:
# ============================================================
# 加载 Qwen 模型（通过 ModelScope）
# ============================================================
# 如果没有 GPU 或显存不足，可将模型 ID 改为 Qwen/Qwen2.5-3B-Instruct
# ============================================================

import torch
from modelscope import AutoModelForCausalLM, AutoTokenizer


class QwenLLM:
    """基于 ModelScope 的 Qwen 模型封装类"""

    def __init__(self, model_name="Qwen/Qwen2.5-7B-Instruct", device=None):
        # 自动检测 GPU / CPU
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        print(f"[QwenLLM] 使用设备: {self.device}")
        print(f"[QwenLLM] 加载模型: {model_name}")

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype="auto", device_map="auto"
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.messages = []
        print("[QwenLLM] 模型加载完成！")

    def chat(self, user_message, system_prompt=None, max_new_tokens=512, temperature=0.7):
        """调用模型进行对话"""
        if system_prompt:
            messages = [{"role": "system", "content": system_prompt}]
        else:
            messages = []
        messages.extend(self.messages)
        messages.append({"role": "user", "content": user_message})

        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, do_sample=True
        )
        generated_ids = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        self.messages.append({"role": "user", "content": user_message})
        self.messages.append({"role": "assistant", "content": response})
        return response

    def reset(self):
        """重置对话历史"""
        self.messages = []


# 初始化 QwenLLM 实例
llm = QwenLLM(model_name="Qwen/Qwen2.5-7B-Instruct")
print("\n模型就绪，可以开始使用了。")

# 00 - 多 Agent 系统设计与协作模式

## 学习目标

- 理解多 Agent 系统的核心设计原则
- 掌握常见的 Agent 协作模式
- 学习 Agent 通信与协调机制
- 实现一个完整的多 Agent 协作系统

---

## 1. 多 Agent 系统概述

### 1.1 什么是多 Agent 系统？

**多 Agent 系统（Multi-Agent System, MAS）** 是由多个自主 Agent 组成的系统，这些 Agent 通过协作、竞争或协商来完成复杂任务。

**核心特征**：

| 特征 | 说明 |
|------|------|
| **自主性** | 每个 Agent 独立决策和行动 |
| **分布性** | Agent 分布在不同节点或进程中 |
| **协作性** | Agent 之间可以协作完成任务 |
| **适应性** | 系统能适应环境变化 |
| **涌现性** | 整体行为大于个体之和 |

### 1.2 多 Agent 系统的优势

```
单 Agent 系统                    多 Agent 系统
┌─────────────┐                ┌─────────────┐
│             │                │  Agent A    │
│   Agent     │                ├─────────────┤
│  (全能型)   │                │  Agent B    │
│             │                ├─────────────┤
└─────────────┘                │  Agent C    │
                               └─────────────┘
                                    │
                              ┌─────┴─────┐
                              ▼           ▼
                         分工协作      并行处理
```

**优势**：
- **模块化**：每个 Agent 专注特定领域
- **可扩展性**：容易添加新 Agent
- **容错性**：单个 Agent 失败不影响整体
- **并行性**：多个任务同时处理
- **专业化**：每个 Agent 可以深度优化

---

## 2. Agent 协作模式

### 2.1 常见协作模式

#### 模式一：层级结构（Hierarchical）

```
         ┌─────────┐
         │ 管理者   │
         │ Agent   │
         └────┬────┘
              │
      ┌───────┼───────┐
      ▼       ▼       ▼
  ┌──────┐┌──────┐┌──────┐
  │执行A ││执行B ││执行C │
  └──────┘└──────┘└──────┘
```

**特点**：
- 一个管理者 Agent 协调多个执行 Agent
- 适合任务分解和分配
- 决策集中，执行分散

#### 模式二：对等网络（Peer-to-Peer）

```
      ┌─────┐
      │  A  │◄──────┐
      └──┬──┘       │
         │          │
    ┌────┴────┐     │
    ▼         ▼     │
 ┌─────┐   ┌─────┐  │
 │  B  │◄─►│  C  │──┘
 └─────┘   └─────┘
```

**特点**：
- Agent 之间平等协作
- 通过消息传递通信
- 适合分布式问题解决

#### 模式三：流水线（Pipeline）

```
  ┌─────┐   ┌─────┐   ┌─────┐   ┌─────┐
  │输入  │──►│处理A │──►│处理B │──►│输出  │
  └─────┘   └─────┘   └─────┘   └─────┘
```

**特点**：
- 数据流驱动
- 每个 Agent 处理特定阶段
- 适合数据处理工作流

#### 模式四：市场机制（Market-based）

```
  任务发布                    任务竞标
  ┌─────┐                   ┌─────┐
  │任务A │◄─────────────────│Agent1│
  ├─────┤                   ├─────┤
  │任务B │◄─────────────────│Agent2│
  ├─────┤                   ├─────┤
  │任务C │◄─────────────────│Agent3│
  └─────┘                   └─────┘
```

**特点**：
- Agent 竞标任务
- 基于能力匹配分配
- 适合动态负载均衡

### 2.2 模式对比

| 模式 | 通信复杂度 | 容错性 | 扩展性 | 适用场景 |
|------|-----------|--------|--------|----------|
| 层级结构 | 低 | 中 | 中 | 任务分解 |
| 对等网络 | 高 | 高 | 高 | 分布式问题 |
| 流水线 | 低 | 低 | 中 | 数据处理 |
| 市场机制 | 中 | 高 | 高 | 动态分配 |

---

## 3. Agent 通信机制

### 3.1 通信类型

```python
# 1. 直接通信（Direct Communication）
agent_a.send_message(agent_b, "任务完成")

# 2. 广播通信（Broadcast）
agent_a.broadcast("新任务可用")

# 3. 黑板系统（Blackboard）
blackboard.write("shared_data", data)
data = blackboard.read("shared_data")

# 4. 消息队列（Message Queue）
queue.publish("task_queue", task)
task = queue.subscribe("task_queue")
```

### 3.2 通信协议设计

```python
class Message:
    """Agent 间消息"""
    
    def __init__(self, sender, receiver, msg_type, content, timestamp=None):
        self.sender = sender
        self.receiver = receiver  # None 表示广播
        self.msg_type = msg_type
        self.content = content
        self.timestamp = timestamp or time.time()
    
    def __repr__(self):
        return f"Message({self.sender} -> {self.receiver}: {self.msg_type})"
```

**消息类型**：

| 类型 | 用途 |
|------|------|
| `TASK_ASSIGN` | 任务分配 |
| `TASK_COMPLETE` | 任务完成通知 |
| `REQUEST_HELP` | 请求协助 |
| `SHARE_INFO` | 信息共享 |
| `COORDINATE` | 协调请求 |

---

## 4. 动手实现：多 Agent 协作系统

下面我们实现一个支持多种协作模式的多 Agent 系统：

In [ ]:
import time
import random
from enum import Enum
from typing import List, Dict, Optional
from dataclasses import dataclass, field

class MessageType(Enum):
    TASK_ASSIGN = "task_assign"
    TASK_COMPLETE = "task_complete"
    REQUEST_HELP = "request_help"
    SHARE_INFO = "share_info"
    COORDINATE = "coordinate"
    BROADCAST = "broadcast"

@dataclass
class Message:
    """Agent 间消息"""
    sender: str
    receiver: Optional[str]  # None 表示广播
    msg_type: MessageType
    content: Dict
    timestamp: float = field(default_factory=time.time)
    
    def __repr__(self):
        target = self.receiver or "ALL"
        return f"📨 [{self.sender} -> {target}] {self.msg_type.value}"

# 测试消息
msg = Message(
    sender="agent_1",
    receiver="agent_2",
    msg_type=MessageType.TASK_ASSIGN,
    content={"task_id": "task_001", "description": "研究 AI 趋势"}
)
print(msg)
print(f"内容: {msg.content}")

In [ ]:
class MessageBus:
    """消息总线 - 负责 Agent 间通信"""
    
    def __init__(self):
        self.messages: List[Message] = []
        self.subscribers: Dict[str, List[str]] = {}  # agent_id -> [msg_types]
        self.message_history: List[Message] = []
    
    def send(self, message: Message):
        """发送消息"""
        self.messages.append(message)
        self.message_history.append(message)
        print(f"📤 发送: {message}")
    
    def get_messages_for(self, agent_id: str) -> List[Message]:
        """获取指定 Agent 的消息"""
        # 获取直接发送给该 Agent 的消息或广播消息
        relevant = [
            msg for msg in self.messages
            if msg.receiver == agent_id or msg.receiver is None
        ]
        # 从消息队列中移除已获取的消息
        for msg in relevant:
            if msg in self.messages:
                self.messages.remove(msg)
        return relevant
    
    def broadcast(self, sender: str, msg_type: MessageType, content: Dict):
        """广播消息"""
        msg = Message(sender=sender, receiver=None, msg_type=msg_type, content=content)
        self.send(msg)
    
    def get_history(self) -> List[Message]:
        """获取消息历史"""
        return self.message_history

# 测试消息总线
bus = MessageBus()

# 发送几条消息
bus.send(Message("agent_1", "agent_2", MessageType.TASK_ASSIGN, {"task": "研究"}))
bus.broadcast("agent_2", MessageType.TASK_COMPLETE, {"result": "完成"})
bus.send(Message("agent_3", "agent_1", MessageType.SHARE_INFO, {"info": "新数据"}))

print("\n📥 agent_1 收到的消息:")
for msg in bus.get_messages_for("agent_1"):
    print(f"  {msg}")

In [ ]:
import time
import random
from enum import Enum
from typing import List, Dict, Optional
from dataclasses import dataclass, field

class MessageType(Enum):
    TASK_ASSIGN = "task_assign"
    TASK_COMPLETE = "task_complete"
    REQUEST_HELP = "request_help"
    SHARE_INFO = "share_info"
    COORDINATE = "coordinate"
    BROADCAST = "broadcast"

@dataclass
class Message:
    """Agent 间消息"""
    sender: str
    receiver: Optional[str]  # None 表示广播
    msg_type: MessageType
    content: Dict
    timestamp: float = field(default_factory=time.time)
    
    def __repr__(self):
        target = self.receiver or "ALL"
        return f"Message({self.sender} -> {target}: {self.msg_type.value})"

# 测试消息
msg = Message(
    sender="agent_1",
    receiver="agent_2",
    msg_type=MessageType.TASK_ASSIGN,
    content={"task_id": "task_001", "description": "研究 AI 趋势"}
)
print(msg)
print(f"内容: {msg.content}")

In [ ]:
class MultiAgentSystem:
    """多 Agent 系统（使用 QwenLLM）"""
    
    def __init__(self, name: str, llm_model=None):
        self.name = name
        self.agents: Dict[str, CollaborativeAgent] = {}
        self.message_bus = MessageBus()
        self.task_queue: List[Dict] = []
        self.llm = llm_model  # QwenLLM 实例
    
    def add_agent(self, agent_id: str, name: str, capabilities: List[str]):
        """添加 Agent"""
        agent = CollaborativeAgent(agent_id, name, capabilities, self.message_bus, llm_model=self.llm)
        self.agents[agent_id] = agent
        print(f"添加 Agent: {name} (能力: {capabilities})")
    
    def submit_task(self, task: Dict) -> str:
        """提交任务"""
        task["id"] = f"task_{len(self.task_queue) + 1}"
        task["status"] = "pending"
        self.task_queue.append(task)
        
        # 寻找合适的 Agent
        best_agent = self._find_best_agent(task)
        
        if best_agent:
            task["status"] = "assigned"
            task["assigner"] = "system"
            
            # 发送任务分配消息
            self.message_bus.send(Message(
                sender="system",
                receiver=best_agent.agent_id,
                msg_type=MessageType.TASK_ASSIGN,
                content=task
            ))
            
            # Agent 接收消息并执行
            best_agent.receive_messages()
            
            return f"任务已分配给 {best_agent.name}"
        else:
            return "没有合适的 Agent 可处理此任务"
    
    def _find_best_agent(self, task: Dict) -> Optional[CollaborativeAgent]:
        """寻找最适合处理任务的 Agent"""
        task_type = task.get("type", "general")
        
        available_agents = [
            agent for agent in self.agents.values()
            if agent.can_handle(task_type) and agent.status == "idle"
        ]
        
        if not available_agents:
            return None
        
        # 选择负载最小的 Agent
        return min(available_agents, key=lambda a: a.tasks_completed)
    
    def get_system_status(self) -> Dict:
        """获取系统状态"""
        return {
            "name": self.name,
            "agents": {
                aid: {
                    "name": agent.name,
                    "status": agent.status,
                    "tasks_completed": agent.tasks_completed,
                    "capabilities": agent.capabilities
                }
                for aid, agent in self.agents.items()
            },
            "pending_tasks": len([t for t in self.task_queue if t["status"] == "pending"]),
            "total_tasks": len(self.task_queue)
        }
    
    def print_status(self):
        """打印系统状态"""
        status = self.get_system_status()
        print("\n" + "="*60)
        print(f"系统状态: {status['name']}")
        print("="*60)
        print(f"总任务数: {status['total_tasks']}, 待处理: {status['pending_tasks']}")
        print("\nAgent 状态:")
        for aid, info in status["agents"].items():
            print(f"  {info['name']}: {info['status']} | 已完成: {info['tasks_completed']} | 能力: {info['capabilities']}")

# 创建多 Agent 系统（传入 QwenLLM 实例）
mas = MultiAgentSystem("内容创作团队", llm_model=llm)

# 添加 Agent
mas.add_agent("researcher", "研究员", ["research", "analyze"])
mas.add_agent("writer", "写手", ["write", "review"])
mas.add_agent("analyst", "分析师", ["analyze", "research"])

mas.print_status()

In [ ]:
# 提交任务
tasks = [
    {"type": "research", "description": "研究 AI Agent 最新趋势"},
    {"type": "write", "description": "撰写 AI Agent 趋势文章"},
    {"type": "analyze", "description": "分析市场数据"},
    {"type": "research", "description": "调查竞争对手"},
    {"type": "review", "description": "审核文章质量"},
]

print("\n🚀 开始提交任务...\n")
for task in tasks:
    result = mas.submit_task(task)
    print(f"结果: {result}\n")

mas.print_status()

# 查看消息历史
print("\n📨 消息历史:")
for msg in mas.message_bus.get_history():
    print(f"  {msg}")

---

## 5. 高级协作模式实现

### 5.1 层级协作模式

In [ ]:
class HierarchicalMultiAgentSystem(MultiAgentSystem):
    """层级多 Agent 系统（使用 QwenLLM）"""
    
    def __init__(self, name: str, llm_model=None):
        super().__init__(name, llm_model=llm_model)
        self.manager: Optional[CollaborativeAgent] = None
    
    def set_manager(self, agent_id: str):
        """设置管理者"""
        if agent_id in self.agents:
            self.manager = self.agents[agent_id]
            print(f"设置管理者: {self.manager.name}")
    
    def submit_task(self, task: Dict) -> str:
        """提交任务（层级模式）"""
        if not self.manager:
            return "未设置管理者"
        
        task["id"] = f"task_{len(self.task_queue) + 1}"
        self.task_queue.append(task)
        
        # 管理者分解任务
        print(f"\n[{self.manager.name}] 接收任务: {task['description']}")
        subtasks = self._decompose_task(task)
        
        print(f"   分解为 {len(subtasks)} 个子任务:")
        results = []
        for i, subtask in enumerate(subtasks, 1):
            print(f"   子任务 {i}: {subtask['description']}")
            
            # 为子任务找到合适的 Agent
            agent = self._find_best_agent(subtask)
            if agent:
                subtask["assigner"] = self.manager.agent_id
                result = agent.assign_task(subtask)
                results.append(result)
            else:
                results.append(f"无法执行: {subtask['description']}")
        
        # 管理者整合结果
        final_result = self._integrate_results(results)
        return final_result
    
    def _decompose_task(self, task: Dict) -> List[Dict]:
        """分解任务"""
        description = task["description"]
        
        return [
            {"id": f"{task.get('id', 'task')}_1", "type": "research", "description": f"研究: {description}", "assigner": self.manager.agent_id},
            {"id": f"{task.get('id', 'task')}_2", "type": "analyze", "description": f"分析: {description}", "assigner": self.manager.agent_id},
            {"id": f"{task.get('id', 'task')}_3", "type": "write", "description": f"总结: {description}", "assigner": self.manager.agent_id},
        ]
    
    def _integrate_results(self, results: List[str]) -> str:
        """使用 QwenLLM 整合结果"""
        print(f"\n[{self.manager.name}] 整合结果:")
        summary = "\n".join([f"  - {r}" for r in results])
        print(summary)

        # 使用 QwenLLM 生成整合摘要
        if self.llm is not None:
            system_prompt = "你是一个项目管理专家，请根据以下子任务结果，生成简洁的整合摘要（100字以内）。"
            integrated = self.llm.chat(f"子任务结果:\n{summary}", system_prompt=system_prompt, max_new_tokens=200)
            return f"\n任务完成！\n整合摘要:\n{integrated.strip()}"
        else:
            # ---- 无模型时的备选方案 ----
            return f"\n任务完成！\n整合结果:\n{summary}"

# 创建层级系统（传入 QwenLLM 实例）
hmas = HierarchicalMultiAgentSystem("项目管理系统", llm_model=llm)

# 添加 Agent
hmas.add_agent("manager", "项目经理", ["research", "analyze", "write"])
hmas.add_agent("researcher", "研究员", ["research"])
hmas.add_agent("analyst", "分析师", ["analyze"])
hmas.add_agent("writer", "写手", ["write"])

# 设置管理者
hmas.set_manager("manager")

# 提交复杂任务
result = hmas.submit_task({
    "description": "完成 AI Agent 市场调研报告"
})
print(result)

### 5.2 市场机制协作模式

In [ ]:
class MarketBasedSystem(MultiAgentSystem):
    """基于市场机制的多 Agent 系统（使用 QwenLLM）"""
    
    def __init__(self, name: str, llm_model=None):
        super().__init__(name, llm_model=llm_model)
        self.agent_reputation: Dict[str, float] = {}
    
    def add_agent(self, agent_id: str, name: str, capabilities: List[str]):
        super().add_agent(agent_id, name, capabilities)
        self.agent_reputation[agent_id] = 1.0  # 初始声誉值
    
    def submit_task(self, task: Dict) -> str:
        """提交任务（竞标模式）"""
        task["id"] = f"task_{len(self.task_queue) + 1}"
        self.task_queue.append(task)
        
        task_type = task.get("type", "general")
        
        # 收集竞标
        print(f"\n发布任务: {task['description']}")
        print("   开始竞标...\n")
        
        bids = []
        for agent_id, agent in self.agents.items():
            if agent.can_handle(task_type) and agent.status == "idle":
                # 计算竞标分数（基于声誉和负载）
                reputation = self.agent_reputation[agent_id]
                load = agent.tasks_completed
                score = reputation / (load + 1)
                
                bids.append((agent_id, score))
                print(f"   {agent.name} 竞标 - 分数: {score:.2f} (声誉: {reputation}, 负载: {load})")
        
        if not bids:
            return "没有 Agent 竞标此任务"
        
        # 选择最高分的 Agent
        winner_id, winning_score = max(bids, key=lambda x: x[1])
        winner = self.agents[winner_id]
        
        print(f"\n中标: {winner.name} (分数: {winning_score:.2f})")
        
        # 执行任务
        task["assigner"] = "system"
        result = winner.assign_task(task)
        
        # 使用 QwenLLM 评估任务质量（更新声誉）
        if self.llm is not None:
            eval_prompt = f"评估以下任务完成质量，只回答'通过'或'不通过'：\n任务: {task['description']}\n结果: {result[:200]}"
            eval_response = self.llm.chat(eval_prompt, max_new_tokens=50, temperature=0.3)
            success = "通过" in eval_response
        else:
            # ---- 无模型时的备选方案 ----
            success = random.random() > 0.2  # 80% 成功率

        if success:
            self.agent_reputation[winner_id] *= 1.1
            print(f"   {winner.name} 声誉提升: {self.agent_reputation[winner_id]:.2f}")
        else:
            self.agent_reputation[winner_id] *= 0.9
            print(f"   {winner.name} 声誉下降: {self.agent_reputation[winner_id]:.2f}")
        
        return result

# 创建市场机制系统（传入 QwenLLM 实例）
market = MarketBasedSystem("竞标系统", llm_model=llm)

# 添加多个同类型 Agent
market.add_agent("researcher1", "研究员A", ["research", "analyze"])
market.add_agent("researcher2", "研究员B", ["research", "analyze"])
market.add_agent("researcher3", "研究员C", ["research", "analyze"])

# 提交多个任务，观察竞标过程
for i in range(3):
    print("\n" + "="*50)
    market.submit_task({
        "type": "research",
        "description": f"研究任务 {i+1}: AI 趋势分析"
    })

print("\n" + "="*50)
print("最终声誉排名:")
for agent_id, rep in sorted(market.agent_reputation.items(), key=lambda x: x[1], reverse=True):
    agent = market.agents[agent_id]
    print(f"  {agent.name}: {rep:.2f}")

---

## 6. 小结

### 核心要点

1. **多 Agent 系统** 通过协作完成复杂任务，具有模块化、可扩展、容错等优势
2. **协作模式**：层级结构、对等网络、流水线、市场机制
3. **通信机制**：直接通信、广播、黑板系统、消息队列
4. **任务分配**：基于能力匹配、负载均衡、竞标机制
5. **关键挑战**：协调、一致性、通信开销、容错处理

### 下一步

- [01_rag_agent.ipynb](01_rag_agent.ipynb) - 学习 RAG Agent 实现
- [02_tool_use_advanced.ipynb](02_tool_use_advanced.ipynb) - 高级工具使用

---

## 参考资源

- [Multi-Agent Systems 教材](http://www.masfoundations.org/)
- [AutoGen 多 Agent 设计](https://microsoft.github.io/autogen/)
- [CrewAI 协作模式](https://docs.crewai.com/)
- [LangGraph 多 Agent 工作流](https://langchain-ai.github.io/langgraph/)